# EDA — Despesas Públicas FACAPE (2021–2025)

**Objetivo:** entender o dataset antes de modelar.  
**Pergunta de negócio:** *Quais contratos têm risco de execução problemática?*

Etapas:
1. Carregamento e limpeza
2. Evolução temporal dos gastos
3. Perfil de fornecedores
4. Distribuição dos valores
5. Execução orçamentária (taxa empenhado → pago)
6. Análise do target (contratos problemáticos)
7. Conclusões para a modelagem

In [ ]:
import sys
from pathlib import Path

# Garante que o root do projeto está no path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.load_data import load_jsons

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

print('Setup OK')

## 1. Carregamento dos Dados

In [ ]:
df = load_jsons()

print(f'Shape: {df.shape}')
print(f'Anos:  {sorted(df["ano"].unique())}')
print(f'Tipos: PF={( df["tipo"]==0).sum()} | PJ={(df["tipo"]==1).sum()}')
print(f'\nColunas: {list(df.columns)}')

In [ ]:
# Estatísticas descritivas dos valores financeiros
cols_valor = ['valor_empenhado', 'valor_pago', 'valor_anulado', 'valor_retido', 'taxa_execucao', 'perc_anulado']
df[cols_valor].describe().round(3)

In [ ]:
# Registros por ano
print('Registros por ano:')
print(df.groupby('ano').size().rename('n_contratos').to_frame())

## 2. Evolução Temporal dos Gastos

In [ ]:
resumo_ano = (
    df.groupby('ano')[['valor_empenhado', 'valor_pago']]
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1 — Evolução dos pagamentos
sns.lineplot(data=resumo_ano, x='ano', y='valor_pago',
             marker='o', linewidth=2.5, color='steelblue', ax=axes[0])
for _, row in resumo_ano.iterrows():
    axes[0].text(row['ano'], row['valor_pago'] + 300_000,
                 f'R$ {row["valor_pago"]/1_000_000:.1f}M', ha='center', fontsize=9)
axes[0].set_title('Evolução dos Pagamentos por Ano')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Total Pago (R$)')

# Gráfico 2 — Empenhado vs Pago
resumo_melted = resumo_ano.melt(
    id_vars='ano', value_vars=['valor_empenhado', 'valor_pago'],
    var_name='tipo', value_name='valor'
)
resumo_melted['tipo'] = resumo_melted['tipo'].map({
    'valor_empenhado': 'Empenhado', 'valor_pago': 'Pago'
})
sns.barplot(data=resumo_melted, x='ano', y='valor', hue='tipo',
            palette=['steelblue', 'seagreen'], ax=axes[1])
axes[1].set_title('Empenhado vs Pago por Ano')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Valor (R$)')
axes[1].legend(title='')

plt.tight_layout()
plt.savefig(REPORTS / 'eda_01_evolucao_temporal.png', dpi=150)
plt.show()

print('Observação: gap entre empenhado e pago indica contratos não executados integralmente.')

## 3. Perfil de Fornecedores

In [ ]:
# Top 10 fornecedores por valor total pago
top10 = (
    df.groupby('fornecedor')['valor_pago']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.barplot(data=top10, x='valor_pago', y='fornecedor', palette='Blues_r')
for i, row in top10.iterrows():
    plt.text(row['valor_pago'] + 100_000, i,
             f'R$ {row["valor_pago"]/1_000_000:.1f}M', va='center', fontsize=9)
plt.title('Top 10 Fornecedores por Valor Total Pago (2021–2025)')
plt.xlabel('Total Pago (R$)')
plt.ylabel('')
plt.tight_layout()
plt.savefig(REPORTS / 'eda_02_top10_fornecedores.png', dpi=150)
plt.show()

In [ ]:
# Fornecedores novos vs recorrentes por ano
contagem_por_ano = df.groupby('ano')['fornecedor'].nunique().reset_index()
contagem_por_ano.columns = ['ano', 'total_fornecedores']

novos_por_ano = {}
vistos = set()
for ano in sorted(df['ano'].unique()):
    fornecedores_ano = set(df[df['ano'] == ano]['fornecedor'].dropna())
    novos_por_ano[ano] = len(fornecedores_ano - vistos)
    vistos.update(fornecedores_ano)

df_novos = pd.DataFrame(novos_por_ano.items(), columns=['ano', 'novos'])
df_novos = df_novos.merge(contagem_por_ano, on='ano')
df_novos['recorrentes'] = df_novos['total_fornecedores'] - df_novos['novos']

plt.figure(figsize=(9, 5))
plt.bar(df_novos['ano'], df_novos['recorrentes'], label='Recorrentes', color='steelblue')
plt.bar(df_novos['ano'], df_novos['novos'], bottom=df_novos['recorrentes'],
        label='Novos', color='coral')
plt.title('Fornecedores Novos vs Recorrentes por Ano')
plt.xlabel('Ano')
plt.ylabel('Quantidade')
plt.legend()
plt.tight_layout()
plt.savefig(REPORTS / 'eda_03_novos_vs_recorrentes.png', dpi=150)
plt.show()

print('Observação: maioria dos fornecedores é nova a cada ano — poucos são parceiros recorrentes.')

## 4. Distribuição dos Valores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição raw (com outliers)
df_pos = df[df['valor_pago'] > 0]
sns.histplot(df_pos['valor_pago'], bins=40, color='steelblue', kde=True, ax=axes[0])
axes[0].set_title('Distribuição dos Valores Pagos')
axes[0].set_xlabel('Valor Pago (R$)')
axes[0].set_ylabel('Contratos')

# Distribuição em log scale (mais legível)
sns.histplot(np.log1p(df_pos['valor_pago']), bins=40, color='seagreen', kde=True, ax=axes[1])
axes[1].set_title('Distribuição em Escala Log')
axes[1].set_xlabel('log(1 + Valor Pago)')
axes[1].set_ylabel('Contratos')

plt.suptitle('A distribuição é altamente assimétrica — poucos contratos concentram a maior parte do valor')
plt.tight_layout()
plt.savefig(REPORTS / 'eda_04_distribuicao_valores.png', dpi=150)
plt.show()

p50 = df_pos['valor_pago'].quantile(0.50)
p90 = df_pos['valor_pago'].quantile(0.90)
print(f'Mediana: R$ {p50:,.0f} | P90: R$ {p90:,.0f} | Máximo: R$ {df_pos["valor_pago"].max():,.0f}')
print('→ Usar log(valor_empenhado) como feature nos modelos (já implementado em src/features.py)')

## 5. Execução Orçamentária

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Taxa de execução por ano
taxa_por_ano = df.groupby('ano')['taxa_execucao'].mean().reset_index()
sns.barplot(data=taxa_por_ano, x='ano', y='taxa_execucao', palette='Greens', ax=axes[0])
axes[0].axhline(y=1.0, color='red', linestyle='--', linewidth=1.2, label='100%')
for i, row in taxa_por_ano.iterrows():
    axes[0].text(i, row['taxa_execucao'] + 0.01, f'{row["taxa_execucao"]*100:.1f}%',
                 ha='center', fontsize=10)
axes[0].set_title('Taxa de Execução Média por Ano')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Taxa Média (pago/empenhado)')
axes[0].set_ylim(0, 1.2)
axes[0].legend()

# Anulação e retenção por tipo de fornecedor
tipo_stats = (
    df.groupby('tipo')[['perc_anulado', 'perc_retido']]
    .mean()
    .reset_index()
)
tipo_stats['tipo'] = tipo_stats['tipo'].map({0: 'Pessoa Física', 1: 'Pessoa Jurídica'})
tipo_melted = tipo_stats.melt(id_vars='tipo', var_name='metrica', value_name='percentual')
tipo_melted['metrica'] = tipo_melted['metrica'].map({
    'perc_anulado': '% Anulado', 'perc_retido': '% Retido'
})
sns.barplot(data=tipo_melted, x='tipo', y='percentual', hue='metrica',
            palette=['coral', 'steelblue'], ax=axes[1])
axes[1].set_title('Anulação e Retenção por Tipo de Fornecedor')
axes[1].set_xlabel('')
axes[1].set_ylabel('Percentual médio')

plt.tight_layout()
plt.savefig(REPORTS / 'eda_05_execucao_orcamentaria.png', dpi=150)
plt.show()

## 6. Análise do Target — Contratos Problemáticos

**Definição:** contrato problemático = `taxa_execucao < 70%` **OU** `valor_anulado > 0`

In [ ]:
df['target'] = ((df['taxa_execucao'] < 0.70) | (df['valor_anulado'] > 0)).astype(int)

print(f'Contratos problemáticos: {df["target"].sum()} ({df["target"].mean():.1%})')
print(f'Contratos OK:            {(df["target"]==0).sum()} ({(df["target"]==0).mean():.1%})')
print('\n→ Dataset razoavelmente balanceado — não precisa de SMOTE ou class_weight extremo')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# % problemáticos por ano
target_ano = df.groupby('ano')['target'].mean().reset_index()
target_ano['pct'] = target_ano['target'] * 100
sns.barplot(data=target_ano, x='ano', y='pct', color='coral', ax=axes[0])
for i, row in target_ano.iterrows():
    axes[0].text(i, row['pct'] + 0.5, f'{row["pct"]:.1f}%', ha='center', fontsize=10)
axes[0].set_title('% de Contratos Problemáticos por Ano')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('% Problemáticos')
axes[0].set_ylim(0, 80)

# % problemáticos por tipo
target_tipo = df.groupby('tipo')['target'].mean().reset_index()
target_tipo['tipo'] = target_tipo['tipo'].map({0: 'Pessoa Física', 1: 'Pessoa Jurídica'})
target_tipo['pct'] = target_tipo['target'] * 100
sns.barplot(data=target_tipo, x='tipo', y='pct', palette=['steelblue', 'seagreen'], ax=axes[1])
for i, row in target_tipo.iterrows():
    axes[1].text(i, row['pct'] + 0.5, f'{row["pct"]:.1f}%', ha='center', fontsize=11)
axes[1].set_title('% de Contratos Problemáticos por Tipo')
axes[1].set_xlabel('')
axes[1].set_ylabel('% Problemáticos')
axes[1].set_ylim(0, 80)

plt.tight_layout()
plt.savefig(REPORTS / 'eda_06_target_analise.png', dpi=150)
plt.show()

In [ ]:
# Correlação entre features numéricas e o target
feature_cols = ['valor_empenhado', 'taxa_execucao', 'perc_anulado', 'perc_retido', 'perc_saldo', 'tipo']

corr = df[feature_cols + ['target']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Correlação entre Features e Target')
plt.tight_layout()
plt.savefig(REPORTS / 'eda_07_correlacao.png', dpi=150)
plt.show()

print('Correlações com o target:')
print(corr['target'].drop('target').sort_values(key=abs, ascending=False).round(3))

## 7. Conclusões para a Modelagem

| Observação | Implicação para o modelo |
|---|---|
| Distribuição de valores altamente assimétrica | Usar `log1p(valor_empenhado)` como feature |
| 42% dos contratos são problemáticos | Dataset razoavelmente balanceado, F1 é a métrica certa |
| Maioria dos fornecedores aparece só 1 vez | Feature `fornecedor_novo` captura esse sinal |
| `perc_anulado` e `perc_retido` correlacionados com target | Incluídos como features |
| Comportamento varia por ano | Validação out-of-time é essencial (treino 2021–2023, teste 2024–2025) |
| Fornecedores recorrentes têm padrão histórico | Feature de lookback temporal capta esse histórico |

**Próximos passos:** ver `src/train.py` para a modelagem completa com MLflow.